In [2]:
#import shutil
#shutil.rmtree("Regression_Output")

<h3>Table 1 — Daily Model Diagnostics</h3>

<div style="overflow-x:auto;">

| column | dtype | description |
|---|---|---|
| train_day | date | Training day |
| test_day | date | Evaluation day |
| symbol | string | Stock symbol |
| horizon | string | Forecast horizon |
| n_train | int | Number of training observations |
| n_test | int | Number of test observations |
| mse | float | Model mean squared error |
| mse_bench | float | Benchmark mean squared error |
| mse_ratio | float | `mse_bench / mse` |
| mae | float | Mean absolute error |
| r2_oos | float | Out-of-sample R² |
| directional_accuracy | float | Fraction of correct sign predictions |
| directional_accuracy_bench | float | Fraction of correct sign predictions of benchmark |
| residual_mean | float | Mean residual |
| residual_std | float | Residual standard deviation |
| run_id | string | Identifier for features / preprocessing / model |

</div>

<hr>

<h3>Table 2 — Run-Specific Identifiers</h3>

<div style="overflow-x:auto;">

| run_id | model_version | feature_set_id | normalization_id |
|---|---|---|---|
| int | string | string | string |

</div>

<hr>

</div>


Add Table to store DM-related outputs. Something like:
Symbol, Trading Day, reg_model_ID, XGBoost_model_ID, LSTM_model_ID, d_reg_XBoost, d_reg_LSTM, ...

In [9]:
import os, warnings
from pathlib import Path
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

import importlib
import utils.data_preprocessing as du
import utils.model_preparation as mu

importlib.reload(du)
importlib.reload(mu)

<module 'utils.model_preparation' from '/home/jovyan/_shared_storage/temp_stud/MA_Otto/utils/model_preparation.py'>

In [10]:
SAVE_TICK_LEVEL_DATA = False

MODEL = "OLS"
MODEL_VERSION = "ols_v1"
FEATURE_SET_ID = "microP_l1Imb_9Lags"
NORMALIZATION_ID = ""

OUTPUT_ROOT = "Regression_Output"

In [13]:
t = pd.read_parquet(f"Feature_Targets/{du.SAMPLE_DATES[0]}/{du.SYMBOLS[0]}.parquet")

In [17]:
# ============================================================
# Initialize Run
# ============================================================
warnings.filterwarnings("ignore")

RUN_ID = mu.initialize_run(
    output_root=OUTPUT_ROOT,
    model_version=MODEL_VERSION,
    feature_set_id=FEATURE_SET_ID,
    normalization_id=NORMALIZATION_ID
)

DIRS = mu.create_output_dirs(
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
)

TICK_OUTPUT_DIR = DIRS["tick"]
DAILY_OUTPUT_DIR = DIRS["daily"]

daily_results = []
ols_results = []

sample_cols = pd.read_parquet(f"Feature_Targets/{du.SAMPLE_DATES[0]}/{du.SYMBOLS[0]}.parquet").columns

FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = list(sample_cols[sample_cols.str.startswith("T_")])

# ============================================================
# Main Loop
# ============================================================

GLOBAL_START = time.perf_counter()

# Load all days for one symbol at once
for day in tqdm(du.SAMPLE_DATES[:10], desc="Processing days", unit="day"):
    load_start = time.perf_counter()

    symbol_cache = {}

    for symbol in du.SYMBOLS[:-1]:
        df = pd.read_parquet(f"Feature_Targets/{day}/{symbol}.parquet").sort_values("Timestamp")
        df = df.dropna(subset=FEATURE_COLS+TARGET_COLS)

        features = df[FEATURE_COLS].to_numpy(dtype=np.float32)

        if NORMALIZATION_ID:
            feature_mean = features.mean(axis=0, dtype=np.float64).astype(np.float32)
            feature_std = features.std(axis=0, dtype=np.float64).astype(np.float32)
            feature_std[feature_std == 0] = 1.0

        symbol_cache[day] = {
            "X": features,
            "Y": df[TARGET_COLS].to_numpy(np.float32),

            "X_mean": feature_mean if NORMALIZATION_ID else None,
            "X_std": feature_std if NORMALIZATION_ID else None,
        
            "timestamp": df["Timestamp"].to_numpy(),
        
            #"relative_spread": df["RelativeSpread"].to_numpy(dtype=np.float32),
            #"midprice": df["MidPrice"].to_numpy(dtype=np.float32),        
            #"l1_imbalance": df["L1-QImb"].to_numpy(dtype=np.float32),
        }

    tqdm.write(f"{symbol}: Loaded all days in {time.perf_counter()-load_start:.2f}s")

    # --------------------------------------------------------
    # Rolling train/test
    # --------------------------------------------------------

    for i in tqdm(range(len(SD) - 1), desc=f"{symbol}", leave=False, unit="split"):

        train_day = SD[i]
        test_day = SD[i+1]

        train_cache = symbol_cache[train_day]
        test_cache = symbol_cache[test_day]
                
        X_train = train_cache["X"]
        X_test = test_cache["X"]

        Y_train = train_cache["Y"]
        Y_test = test_cache["Y"]

        if NORMALIZATION_ID:
            X_train = (train_cache["X"] - train_cache["X_mean"]) / train_cache["X_std"]
            X_test = (test_cache["X"] - train_cache["X_mean"]) / train_cache["X_std"]

        model = LinearRegression()
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test).astype(np.float32)
        
        # Residuals
        RESID = (Y_test - Y_pred).astype(np.float32)

        if SAVE_TICK_LEVEL_DATA:
            tick_df = pd.DataFrame({
                "timestamp": test_cache["timestamp"],
                #"relative_spread": test_cache["relative_spread"],
                #"midprice": test_cache["midprice"],
                #"l1_imbalance": test_cache["l1_imbalance"],
            })
            
            for j, target in enumerate(TARGET_COLS):
                tick_df[f"{target}_y_true"] = Y_test[:, j]
                tick_df[f"{target}_resid"] = RESID[:, j]
                tick_df[f"{target}_resid_bench"] = Y_test[:, j]

            mu.save_table(
                df=tick_df,
                root_dir=TICK_OUTPUT_DIR,
                filename="tick_predictions.feather",
                partition_cols={
                    "date": test_day,
                    "symbol": symbol
                },
                file_format="feather"
            )
        
        mse = np.mean(RESID ** 2, axis=0)
        mse_bench = np.mean(Y_test ** 2, axis=0)
        
        mse_ratio = np.divide(
            mse_bench,
            mse,
            out=np.full_like(mse, np.nan),
            where=mse > 0
        )

        r2_oos = np.divide(
            mse,
            mse_bench,
            out=np.full_like(mse, np.nan),
            where=mse_bench > 0
        )
        
        r2_oos = 1.0 - r2_oos

        for j, target in enumerate(TARGET_COLS):
            daily_results.append({
                "train_day": train_day,
                "test_day": test_day,
                "symbol": symbol,
                "target": target,
                "run_id": RUN_ID,
                "mse": float(mse[j]),
                "mse_bench": float(mse_bench[j]),
                "mse_ratio": float(mse_ratio[j]),
                "r2_oos": float(r2_oos[j]),
            })

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")
# ============================================================
# Save Outputs
# ============================================================

mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=DAILY_OUTPUT_DIR,
    filename="daily_diagnostics.parquet",
    file_format="parquet"
)

Processing days:   0%|          | 0/10 [00:00<?, ?day/s]

Adidas: Loaded all days in 14.71s


Adidas:   0%|          | 0/126 [00:00<?, ?split/s]

KeyError: '2023-01-03'

In [5]:
daily_out[daily_out["mse_ratio"]<1]

NameError: name 'daily_out' is not defined

In [ ]:
len(daily_out[daily_out["mse_ratio"]<1]) / len(daily_out)

In [ ]:
daily_out_1 = pd.read_parquet("Regression_Output/DailyDiagnostics/1/daily_diagnostics.parquet")
daily_out_1